The full_feature_matrix was generated by integrating compositional, structural and experimental descriptors. 
- Removal of non-numeric variables
- Exclusiion of columns with more than 50% missing values 
- Median imputation of remaining missing data 
- Removal of near-constant features using s standard deviation threshold (< 1×10⁻⁶)
- Correlation filtering using a Pearson correlation threshold rate to improve target distribution for ML. 

In [0]:
import pandas as pd
import numpy as np

INPUT_FILE = "full_descriptors_dataset_db_ml.csv"
OUTPUT_FILE = "full_feature_matrix.csv"

# load dataset
df = pd.read_csv(INPUT_FILE)

# -----------------------------
# 1. remove obvious non-features
# -----------------------------
drop_cols = [
    "compound",
    "doi_reference"
]

drop_cols = [c for c in drop_cols if c in df.columns]

df = df.drop(columns=drop_cols)

# remove unit columns
uom_cols = [c for c in df.columns if "uom" in c.lower()]
df = df.drop(columns=uom_cols)

# -----------------------------
# 2. keep only numeric columns
# -----------------------------
df = df.select_dtypes(include=["number"])

# -----------------------------
# 3. remove columns with >50% NaN
# -----------------------------
nan_frac = df.isna().mean()

df = df.loc[:, nan_frac < 0.5]

# -----------------------------
# 4. median imputation
# -----------------------------
df = df.fillna(df.median())

# -----------------------------
# 5. remove near-constant features
# -----------------------------
low_var_cols = []

for col in df.columns:
    if df[col].std() < 1e-6:
        low_var_cols.append(col)

df = df.drop(columns=low_var_cols)

# -----------------------------
# 6. correlation filtering
# -----------------------------
corr_matrix = df.corr().abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

to_drop = [
    column for column in upper.columns
    if any(upper[column] > 0.95)
]

df = df.drop(columns=to_drop)

# -----------------------------
# 7. log-transform target
# -----------------------------
if "hydrogen_production_rate" in df.columns:
    df["log_hydrogen_production_rate"] = np.log10(
        df["hydrogen_production_rate"] + 1
    )

# -----------------------------
# save
# -----------------------------
df.to_csv(OUTPUT_FILE, index=False)

print("Feature engineering completed.")
print("Final shape:", df.shape)
print(f"Saved to {OUTPUT_FILE}")

In [0]:
output_df = pd.read_csv("full_feature_matrix.csv")
display(output_df)

Import check if the descriptors below survived: 

In [0]:
important = [
    "tolerance_factor",
    "octahedral_factor",
    "bandgap_energy",
    "en_pauling_mean",
]

print([c for c in important if c in output_df.columns])

Check which electronegativity descriptor remained

In [0]:
[c for c in output_df.columns if "en_" in c.lower() or "negativity" in c.lower()]

The filter preserved 
- mean electronegativity
- electronegativity spread
- ionization-energy variability

And removed redundant versions like 
- en_pauling_mean

Final feature matrix now still contains the important core physics-informed signals. 